# Akan-BPE Phase 2A1 — Qwen3-0.6B × Akan TTS Tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/2a1_qwen3-0.6b_tts.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/2a1_qwen3-0.6b_tts.ipynb)


Self-contained Colab notebook. Run all cells top-to-bottom.

**Steps:**
1. Clone repo and install deps
2. Download Akan datasets from HuggingFace Hub
3. Run QLoRA fine-tune experiment (`Qwen/Qwen3-0.6B` + Akan TTS tokenizer)
4. Inspect results — fertility reduction, eval loss/perplexity, generation samples

**GPU required.** Before running: Runtime → Change runtime type → T4 GPU.

In [1]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

# Guard against triple-nesting on cell re-run: only clone+cd when we are not
# already sitting inside the repo root.
if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

Cloning into 'akan-bpe'...
remote: Enumerating objects: 377, done.
remote: Counting objects: 100% (377/377), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 377 (delta 220), reused 281 (delta 129), pack-reused 0 (from 0)
Receiving objects: 100% (377/377), 550.72 KiB | 14.49 MiB/s, done.
Resolving deltas: 100% (220/220), done.
/content/akan-bpe
Working directory: /content/akan-bpe


In [2]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.1/513.1 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.3 MB/s eta 0:00:00
  Building editable for akan-bpe (pyproject.toml) ... done


In [3]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU : Tesla T4
VRAM: 15.6 GB


In [4]:
# 4. Download Akan datasets from HuggingFace Hub
# Produces: data/pristine_twi_{train,validation,test}.jsonl
#           data/aka_asr_{train,validation,test}.jsonl
!python scripts/download.py --output-dir data

README.md: 100% 29.2k/29.2k [00:00<00:00, 14.3MB/s]
Resolving data files: 100% 72/72 [00:00<00:00, 272062.96it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 19888.69it/s]
Wrote 8085 rows to data/aka_asr_train.jsonl
Wrote 1011 rows to data/aka_asr_validation.jsonl
Wrote 1011 rows to data/aka_asr_test.jsonl
README.md: 100% 4.06k/4.06k [00:00<00:00, 11.1MB/s]
Wrote 150400 rows to data/pristine_twi_train.jsonl
Wrote 18800 rows to data/pristine_twi_validation.jsonl
Wrote 18800 rows to data/pristine_twi_test.jsonl
Manifest written to data/download_manifest.json


In [5]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found — training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

TTS tokenizer already present (0.5 MB), skipping.


In [6]:
# 5. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data" : Path("data/pristine_twi_train.jsonl"),
    "TTS test data"  : Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer"  : Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

All inputs ready.
  TTS train data: data/pristine_twi_train.jsonl  (240.1 MB)
  TTS test data: data/pristine_twi_test.jsonl  (30.1 MB)
  TTS tokenizer: models/tts_tokenizer.json  (0.5 MB)


In [7]:
# 6. Run Phase 2A1 experiment
# QLoRA fine-tune on Qwen3-0.6B with the Akan TTS tokenizer.
# Writes model adapters to models/phase2a1_qwen3_0_6b_tts/
# Writes result JSON to results/phase2a1_qwen3_0_6b_tts.json
!python scripts/model_integration.py \
    --experiment-id phase2a1_qwen3_0_6b_tts \
    --model-id Qwen/Qwen3-0.6B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a1_qwen3_0_6b_tts \
    --results-output results/phase2a1_qwen3_0_6b_tts.json \
    --device-mode colab-qlora \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

config.json: 100% 726/726 [00:00<00:00, 3.77MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 25.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 114MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:01<00:00, 10.2MB/s]
model.safetensors: 100% 1.50G/1.50G [00:11<00:00, 126MB/s]
Loading weights:   1% 4/311 [00:00<00:14, 21.90it/s, Materializing param=model.layers.0.mlp.down_proj.weight]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 311/311 [00:00<00:00, 340.13it/s, Materializing param=model.norm.weight]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config wi

In [8]:
# 7. Load result JSON
import json
from pathlib import Path

result = json.loads(
    Path("results/phase2a1_qwen3_0_6b_tts.json").read_text(encoding="utf-8")
)
print("Experiment :", result["experiment_id"])
print("Model      :", result["model_id"])
print("Device     :", result.get("device", {}))

Experiment : phase2a1_qwen3_0_6b_tts
Model      : Qwen/Qwen3-0.6B
Device     : {'cuda_available': True, 'device_count': 1, 'device_name': 'Tesla T4'}


In [9]:
# 8. Token count comparison — fertility reduction
cmp = result["token_count_comparison"]
print(f"Base tokenizer fertility : {cmp['base_model_tokenizer']['fertility']:.3f} tokens/word")
print(f"Akan tokenizer fertility : {cmp['experiment_tokenizer']['fertility']:.3f} tokens/word")
print(f"Reduction ratio          : {cmp['token_reduction_ratio']:.1%}")

Base tokenizer fertility : 2.538 tokens/word
Akan tokenizer fertility : 1.280 tokens/word
Reduction ratio          : 49.5%


In [10]:
# 9. Eval metrics
ev = result["eval"]
print(f"Eval loss  : {ev['eval_loss']:.4f}")
print(f"Perplexity : {ev['perplexity']:.2f}")

Eval loss  : 4.4285
Perplexity : 83.81


In [11]:
# 10. Generation samples
for i, s in enumerate(result["generation_samples"], 1):
    print(f"--- Sample {i} ---")
    print(f"Prompt    : {s['prompt']}")
    print(f"Completion: {s['completion']}")
    print()

--- Sample 1 ---
Prompt    : "Ei, Kwame, wotee asɛm a ɛfa BRP ne Motshudi ho no? "
Completion: " Ei , Kwame , wotee asɛm a ɛfa B R P ne Mo ts hu di ho no ? " " Kofi , Kwame , wotee asɛm no ? " " Aane , Kwame , wotee asɛm no ? " " Ɛyɛ nokware . " " Ɛyɛ nokware . " " Ɛyɛ

--- Sample 2 ---
Prompt    : Tete kan, Ananse ne ne mma tenaa adaka bi ase, na wɔn
Completion: Tete kan , Ananse ne ne mma tenaa adaka bi ase , na wɔn ani da a , na ɛyɛ me nwanwa . Ananse kaa sɛ , " Sɛ wopɛ sɛ wopɛ sɛ wopɛ sɛ wopɛ a , ɛsɛ sɛ wopɛ sɛ wopɛ a , ɛsɛ

--- Sample 3 ---
Prompt    : Me nkateɛ ne me adwene wɔ asɛm yi ho yɛ adeɛ a
Completion: Me nkateɛ ne me adwene wɔ asɛm yi ho yɛ adeɛ a ɛyɛ hu . Me , me tirim yɛ me nwanwa . Sɛ me dwene me ho a , me dwene me ho . Me , me dwene me ho yɛ me nwanwa

